In [1]:
import matplotlib.pyplot as plt
import time
import numpy as np
import math

# load test statistics calculation module
import sys, os
sys.path.append(os.path.abspath("../test_stats"))
from test_stat_calc_onnx import *

In [2]:
# load model
model = onnx.load("deepstarr_model.onnx")
graph = model.graph
onnx.checker.check_model(model)
print(onnx.helper.printable_graph(model.graph))

graph tf2onnx (
  %input_12[FLOAT, unk__193x249x4]
) initializers (
  %model_9_1/batch_normalization_65_1/batchnorm/sub:0[FLOAT, 256]
  %model_9_1/batch_normalization_65_1/batchnorm/mul:0[FLOAT, 256]
  %model_9_1/batch_normalization_64_1/batchnorm/sub:0[FLOAT, 256]
  %model_9_1/batch_normalization_64_1/batchnorm/mul:0[FLOAT, 256]
  %model_9_1/batch_normalization_63_1/batchnorm/mul:0[FLOAT, 1x120x1]
  %model_9_1/batch_normalization_62_1/batchnorm/mul:0[FLOAT, 1x60x1]
  %model_9_1/batch_normalization_61_1/batchnorm/mul:0[FLOAT, 1x60x1]
  %model_9_1/batch_normalization_60_1/batchnorm/mul:0[FLOAT, 1x256x1]
  %model_9_1/Dense_Hk_1/Cast/ReadVariableOp:0[FLOAT, 256x1]
  %model_9_1/Dense_Hk_1/Add/ReadVariableOp:0[FLOAT, 1]
  %model_9_1/Dense_Dev_1/Cast/ReadVariableOp:0[FLOAT, 256x1]
  %model_9_1/Dense_Dev_1/Add/ReadVariableOp:0[FLOAT, 1]
  %model_9_1/Dense_2_1/Cast/ReadVariableOp:0[FLOAT, 256x256]
  %model_9_1/Dense_2_1/BiasAdd/ReadVariableOp:0[FLOAT, 256]
  %model_9_1/Dense_1_1/Cast/ReadVaria

In [18]:
# print each layer
prev = None
for node in graph.node:
    if prev != None and prev not in node.input:
        print("Path break")
        print()
    print("-", node.op_type, node.input, node.output)
    prev = node.output[0]

- Unsqueeze ['input_12', 'const_fold_opt__168'] ['model_9_1/Conv1D_1st_1/convolution/ExpandDims:0']
- Transpose ['model_9_1/Conv1D_1st_1/convolution/ExpandDims:0'] ['model_9_1/Conv1D_1st_1/convolution__11:0']
- Conv ['model_9_1/Conv1D_1st_1/convolution__11:0', 'model_9_1/Conv1D_1st_1/convolution/ExpandDims_1:0'] ['model_9_1/Conv1D_1st_1/convolution:0']
- Squeeze ['model_9_1/Conv1D_1st_1/convolution:0', 'const_fold_opt__168__191'] ['model_9_1/Conv1D_1st_1/convolution/Squeeze:0']
- Add ['model_9_1/Conv1D_1st_1/convolution/Squeeze:0', 'const_fold_opt__180'] ['model_9_1/Conv1D_1st_1/BiasAdd:0']
- Mul ['model_9_1/Conv1D_1st_1/BiasAdd:0', 'model_9_1/batch_normalization_60_1/batchnorm/mul:0'] ['model_9_1/batch_normalization_60_1/batchnorm/mul_1:0']
- Add ['model_9_1/batch_normalization_60_1/batchnorm/mul_1:0', 'const_fold_opt__177'] ['model_9_1/batch_normalization_60_1/batchnorm/add_1:0']
- Relu ['model_9_1/batch_normalization_60_1/batchnorm/add_1:0'] ['model_9_1/activation_60_1/Relu:0']
- Un

In [19]:
print("operation count:", len(graph.node))

operation count: 51


In [6]:
# Dense_Dev: developmental enhancers activities
# Dense_Hk: housekeeping enhancers activities
params = {} # get parameters
layer_params = []

extract_params(graph, layer_params, params)

In [13]:
for name, val in params.items():
    print("parameter:", name, "\t parameter shape:", val.shape)

parameter: model_9_1/batch_normalization_65_1/batchnorm/sub:0 	 parameter shape: (256,)
parameter: model_9_1/batch_normalization_65_1/batchnorm/mul:0 	 parameter shape: (256,)
parameter: model_9_1/batch_normalization_64_1/batchnorm/sub:0 	 parameter shape: (256,)
parameter: model_9_1/batch_normalization_64_1/batchnorm/mul:0 	 parameter shape: (256,)
parameter: model_9_1/batch_normalization_63_1/batchnorm/mul:0 	 parameter shape: (1, 120, 1)
parameter: model_9_1/batch_normalization_62_1/batchnorm/mul:0 	 parameter shape: (1, 60, 1)
parameter: model_9_1/batch_normalization_61_1/batchnorm/mul:0 	 parameter shape: (1, 60, 1)
parameter: model_9_1/batch_normalization_60_1/batchnorm/mul:0 	 parameter shape: (1, 256, 1)
parameter: model_9_1/Dense_Hk_1/Cast/ReadVariableOp:0 	 parameter shape: (256, 1)
parameter: model_9_1/Dense_Hk_1/Add/ReadVariableOp:0 	 parameter shape: (1,)
parameter: model_9_1/Dense_Dev_1/Cast/ReadVariableOp:0 	 parameter shape: (256, 1)
parameter: model_9_1/Dense_Dev_1/Add